# Fine-tune Laya for intent-api

Trains Laya (`typed-decisions`) on the generated question set, then compares the tuned checkpoint with the original on the held-out families, and zips the checkpoint.

**Run this on a remote GPU, never on a laptop.** The first code cell stops if there is no GPU.

**Only invented questions go in.** The two data files come from `intent-api training build` and hold no real names, figures or manager questions. Do not upload anything else.

Steps:
1. On your own machine: `uv run --package intent-api intent-api training check` (must print `ok`), then `training build`. It writes `train.jsonl` and `heldout.jsonl` in `apps/intent-api/training/out/`.
2. Put both files next to this notebook on the GPU machine (on Colab the notebook asks you to upload them).
3. Run the cells from top to bottom.
4. Download `intent-tuned.zip`, unzip it into a folder on the machine that runs `laya-serve`, and tell Claude.

Learning curve: build with `training build --fraction 0.25`, then `0.5` and `1.0`, run the notebook once for each and change `RUN_NAME` so the three results stay apart.

In [ ]:
BASE = "typed-decisions"
RUN_NAME = "intent-tuned"
TRAIN_FILE = "train.jsonl"
HELDOUT_FILE = "heldout.jsonl"
EPOCHS = 10
SEED = 0
SHUFFLE_OPTIONS = True

## 1. GPU and Laya

In [ ]:
%pip install -q laya
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU found. Choose a GPU runtime. Do not train on a CPU.")
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 1e9), "GB")

## 2. Data files

In [ ]:
import os

try:
    from google.colab import files
except ImportError:
    files = None

missing = [name for name in (TRAIN_FILE, HELDOUT_FILE) if not os.path.exists(name)]
if missing and files is not None:
    print("Upload", ", ".join(missing))
    files.upload()
for name in (TRAIN_FILE, HELDOUT_FILE):
    assert os.path.exists(name), f"{name} not found next to the notebook"

In [ ]:
import json
from collections import Counter


def read_rows(name):
    with open(name, encoding="utf-8") as file:
        return [json.loads(line) for line in file if line.strip()]


def kind_of(row):
    return "scenario" if "nothing" in row["questions"]["choice"]["criteria"] else "domain"


train_rows = read_rows(TRAIN_FILE)
heldout_rows = read_rows(HELDOUT_FILE)
for name, rows in (("training", train_rows), ("held-out", heldout_rows)):
    kinds = Counter(kind_of(row) for row in rows)
    answers = Counter(row["expected"]["choice"] for row in rows)
    print(f"{name}: {len(rows)} items, {dict(kinds)}")
    print("  answers:", dict(answers.most_common()))
print(json.dumps(train_rows[0], indent=1)[:900])

In [ ]:
from laya.train import TrainConfig, dry_run, finetune, resolve_checkpoint_dir

check = dry_run(TRAIN_FILE, resolve_checkpoint_dir(BASE))
print(check)
assert not check["skipped"], f"some rows cannot be used: {check['skipped']}"

## 3. Train

Soft cross-entropy, options shuffled while training, and `laya-train` sets aside its own calibration slice (10% of the training items, at most 400) to fit the confidence temperature. The held-out file is used only to report before and after numbers.

In [ ]:
config = TrainConfig(
    epochs=EPOCHS,
    loss="soft-ce",
    shuffle_options=("choice",) if SHUFFLE_OPTIONS else (),
    eval_data=HELDOUT_FILE,
    seed=SEED,
)
report = finetune(TRAIN_FILE, BASE, RUN_NAME, config)
print({key: report.get(key) for key in ("train_items", "calibration_items", "skipped", "epoch_loss", "temperature")})
print(report["calibration"])

## 4. Before and after on the held-out families

The held-out families were never trained on. Right and wrong are the exact label Laya chose; confidence is the probability of the chosen label.

In [ ]:
import gc

import laya


def score(model):
    agent = laya.load(model)
    right, total = Counter(), Counter()
    confidence = {True: [], False: []}
    misses = Counter()
    for row in heldout_rows:
        answer = agent.system_one(row["state"], row["questions"])["answers"]["choice"]
        wanted = row["expected"]["choice"]
        ok = answer["choice"] == wanted
        kind = kind_of(row)
        total[kind] += 1
        right[kind] += ok
        confidence[ok].append(answer["probabilities"][answer["choice"]])
        if not ok:
            misses[(wanted, answer["choice"])] += 1
    del agent
    gc.collect()
    torch.cuda.empty_cache()
    return right, total, confidence, misses


def average(values):
    return sum(values) / len(values) if values else 0.0


results = {name: score(model) for name, model in (("before", BASE), ("after", RUN_NAME))}
for kind in ("domain", "scenario"):
    before, after = (results[name][0][kind] / results[name][1][kind] for name in ("before", "after"))
    print(f"{kind:9} accuracy  before {before:.3f}  after {after:.3f}  ({results['after'][1][kind]} items)")
for name in ("before", "after"):
    confidence = results[name][2]
    sure_wrong = sum(value >= 0.7 for value in confidence[False])
    print(f"{name:6} mean confidence right {average(confidence[True]):.2f}, wrong {average(confidence[False]):.2f}; wrong with confidence >= 0.7: {sure_wrong}")
print("most common mistakes after training (wanted, chose, count):")
for (wanted, chose), count in results["after"][3].most_common(10):
    print(f"  {wanted} -> {chose}: {count}")

## 5. Package the checkpoint

In [ ]:
import shutil

archive = shutil.make_archive(RUN_NAME, "zip", RUN_NAME)
print(archive, os.path.getsize(archive) // 1_000_000, "MB")
if files is not None:
    files.download(archive)